# 17 — OE per-pixel inversion with bi_jax (dask engine)

Companion to NB15 (`15_lmfit_superpixel_bi`): replaces the lmfit+superpixel
approach with full per-pixel OE via `oe_engine_optx` tiled through `dask_engine`.

| Layer | NB15 | NB17 |
|---|---|---|
| Layer 1 — solver | `lmfit_engine` (serial LM-LSQ, no priors) | `oe_engine_optx` (JAX vmap OE, with priors) |
| Layer 2 — image  | `superpixel_engine` (~1000 superpixels) | `dask_engine` (all pixels, tiled) |

**Forward model**: `bi_jax` (Bi et al. 2023, HEREON water optical model).  
**Free parameters**: `C_0`–`C_7` (phytoplankton classes), `C_Y` (CDOM), `C_ism` (ISM) — 10 total.  
**Scene**: Helsinki EnMAP L2A.

### Key differences from NB15
- **All water pixels** are inverted (no superpixel compression).
- OE adds **log-normal priors** (`sigma_a`) — retrieval is constrained around
  the prior mean in log-space, preventing runaway concentrations.
- Returns **posterior uncertainty** (`sigma`), **averaging kernel** (`A_diag`),
  and **information content** (`H_info`) per pixel.
- `dask_engine` tiles the image and dispatches each tile to `oe_engine_optx`
  via the shared `invert_fn` interface.

In [ ]:
import numpy as np
import lmfit
import matplotlib.pyplot as plt
import scipy.ndimage as ndi
import time
import xarray as xr
import rioxarray
from pyproj import CRS
from xcube.core.store import new_data_store
import configparser
import jax

from bio_optics.water.reflectance import bi_jax
from bio_optics.inversion import oe_engine, oe_engine_optx
from bio_optics.image_processing import dask_engine

jax.config.update('jax_enable_x64', True)

## Configuration

In [ ]:
NOISE      = 0.001    # Rrs noise level [sr-1] — same as NB15
MAX_STEPS  = 100      # optimistix iteration cap per pixel
TILE_SIZE  = 65536    # pixels per dask tile (default 256×256)
SOLVER     = 'GaussNewton'

## Load scene — Helsinki EnMAP S3

In [ ]:
config = configparser.ConfigParser()
config.read('../../config.ini')
credentials = {k: v.strip() for k, v in config['Credentials'].items()}

store = new_data_store(
    's3', max_depth=5, root='coastal-cubes/sek/',
    storage_options=dict(
        anon=False,
        key=credentials['s3_client_id'],
        secret=credentials['s3_client_secret'],
    )
)

INPUT_PREFIX  = 'helsinki/L2A_land/'
OUTPUT_PREFIX = 'helsinki/temp/'

scene_id = 'ENMAP01-____L2A-DT0000158841_20251019T101424Z_002_V010505_20260206T113145Z'

img         = store.open_data(f'{INPUT_PREFIX}{scene_id}.zarr')
scene_crs   = img.rio.crs or CRS.from_wkt(img.spatial_ref.attrs['crs_wkt'])
wavelengths = img.wavelength.values[:80]

refl  = img['reflectance'].isel(band=slice(0, 80))
rrs   = (refl.where(refl > -32768) / 10_000) / np.pi
cloud = (img['cloud'] == 1) | (img['cirrus'] == 1) | (img['haze'] == 1)
rrs   = rrs.where(~cloud)

_wc = store.open_data(f'{OUTPUT_PREFIX}{scene_id}-worldcover.zarr')
_wf = _wc['water_fraction'].values
_labeled, _ = ndi.label(_wf >= 0.5)
_sizes = np.bincount(_labeled.ravel())
_sizes[0] = 0
ocean_mask = xr.DataArray(
    np.isin(_labeled, np.where(_sizes >= 100)[0]),
    coords=_wc['water_fraction'].coords, dims=_wc['water_fraction'].dims,
)
rrs = rrs.where(ocean_mask)

Rrs_arr = rrs.transpose('y', 'x', 'band').values
n_rows, n_cols, n_obs = Rrs_arr.shape
n_water = int(np.isfinite(Rrs_arr).all(axis=-1).sum())

print(f'Image shape : {Rrs_arr.shape}')
print(f'Water pixels: {n_water}')
print(f'Wavelengths : {wavelengths[0]:.1f} – {wavelengths[-1]:.1f} nm ({n_obs} bands)')

## Parameters and OE setup

Same lmfit Parameters as NB15.  OE additionally requires:
- `sigma_a`: prior width in retrieval space (log-space for log-params).
  A value of 3.0 in log-space means ±2σ spans a factor of ~403 around the prior mean.
- `log_params`: all concentrations are retrieved in log-space (lognormal prior,
  ensures positivity without box constraints).

In [ ]:
params = lmfit.Parameters()

# --- free parameters --------------------------------------------------------
params.add('C_0',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_1',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_2',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_3',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_4',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_5',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_6',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_7',   value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('C_Y',   value=0.2,  min=0.0,   max=10.0,  vary=True)
params.add('C_ism', value=2.0,  min=0.0,   max=200.0, vary=True)
params.add('offset',value=0.0,  min=-0.02, max=0.02,  vary=False)

# --- CDOM -------------------------------------------------------------------
params.add('S_cdom',        value=0.014, vary=False)
params.add('lambda_0_cdom', value=440.0, vary=False)
params.add('K',             value=0.0,   vary=False)

# --- minerogenic detritus absorption ----------------------------------------
params.add('A_md',        value=0.04,  vary=False)
params.add('S_md',        value=0.011, vary=False)
params.add('C_md',        value=0.0,   vary=False)
params.add('lambda_0_md', value=440.0, vary=False)

# --- biogenic detritus absorption -------------------------------------------
params.add('A_bd',        value=0.001, vary=False)
params.add('S_bd',        value=0.011, vary=False)
params.add('C_bd',        value=0.0,   vary=False)
params.add('lambda_0_bd', value=440.0, vary=False)

# --- detritus attenuation ---------------------------------------------------
params.add('gamma_d',     value=0.5,   vary=False)
params.add('x0',          value=0.96,  vary=False)
params.add('x1',          value=0.5,   vary=False)
params.add('x2',          value=1.0,   vary=False)
params.add('lambda_0_c_d',value=550.0, vary=False)

# --- temperature ------------------------------------------------------------
params.add('T_W',   value=15.0, vary=False)
params.add('T_W_0', value=15.0, vary=False)

# --- phytoplankton packaging ------------------------------------------------
params.add('A_phy',        value=0.06,  vary=False)
params.add('E0',           value=0.65,  vary=False)
params.add('E1',           value=0.67,  vary=False)
params.add('lambda_0_phy', value=440.0, vary=False)

# --- backscattering ratios --------------------------------------------------
for i in range(8):
    params.add(f'b_ratio_C_{i}', value=0.01, vary=False)
params.add('b_ratio_md', value=0.02, vary=False)
params.add('b_ratio_bd', value=0.01, vary=False)

# --- Lee et al. (2011) coefficients ----------------------------------------
params.add('Gw0', value=0.0895, vary=False)
params.add('Gw1', value=0.1247, vary=False)
params.add('Gp0', value=0.0401, vary=False)
params.add('Gp1', value=0.0841, vary=False)

# --- OE priors (log-space widths) -------------------------------------------
# sigma_a = 3.0 → ±2σ covers a factor ~403 around the prior mean
# sigma_a = 2.0 → ±2σ covers a factor ~55 around the prior mean
sigma_a = {
    'C_0':   3.0, 'C_1':   3.0, 'C_2':   3.0, 'C_3':   3.0,
    'C_4':   3.0, 'C_5':   3.0, 'C_6':   3.0, 'C_7':   3.0,
    'C_Y':   2.0,
    'C_ism': 3.0,
}
log_params = ['C_0', 'C_1', 'C_2', 'C_3', 'C_4', 'C_5', 'C_6', 'C_7', 'C_Y', 'C_ism']

free = [n for n, p in params.items() if p.vary]
print(f'Free parameters ({len(free)}): {free}')

## Build OE setup

In [ ]:
pre   = bi_jax.precompute(wavelengths)
f_vec = bi_jax.make_forward_vec(list(params.keys()), pre)
setup = oe_engine.build_inversion(params, f_vec, sigma_a, log_params=log_params)

print(f'fit_names : {setup.fit_names}')
print(f'log_params: {log_params}')
print(f'n_obs     : {n_obs}')

## Run — per-pixel OE via dask tiling

`dask_engine.invert_image` tiles the image and calls `oe_engine_optx.invert_image`
per tile via the shared `invert_fn` interface.  JAX vmap runs the full tile in
one compiled call; XLA parallelises across pixels within each tile.

NaN pixels (land, cloud) are ignored inside `oe_engine_optx` and pass NaN through.

## Warm up JAX JIT

`dask_engine` does not handle warmup — JAX compiles on the first `invert_fn` call,
which inflates the first tile's wall-clock time.  Pre-compile here with a tiny
dummy input so the timed run below pays no compilation cost.

`oe_engine_optx` uses `lax.while_loop` (no loop unrolling), so compilation is
**shape-independent**: any pixel count triggers the right kernel.

In [ ]:
print('Warming up JAX JIT ...', end=' ', flush=True)
t_wu = time.perf_counter()
_dummy = np.zeros((2, n_obs))
oe_engine_optx.invert_image(
    _dummy, setup, NOISE,
    solver=SOLVER, max_steps=MAX_STEPS,
    store_chi2_spectral=True, store_y_hat=True,
)
print(f'{time.perf_counter() - t_wu:.1f} s')

In [ ]:
print(f'Inverting {n_water} water pixels ({n_rows}×{n_cols} image, '\n      f'{len(setup.fit_names)} free params, {n_obs} bands) ...')\n\nt0 = time.perf_counter()\nresults = dask_engine.invert_image(\n    Rrs_arr, setup, NOISE,\n    invert_fn=oe_engine_optx.invert_image,\n    tile_size=TILE_SIZE,\n    solver=SOLVER,\n    max_steps=MAX_STEPS,\n    store_chi2_spectral=True,\n    store_y_hat=True,\n)\nt_total = time.perf_counter() - t0\n\nn_tiles = int(np.ceil(n_rows * n_cols / TILE_SIZE))\nprint(f'Done in {t_total:.1f} s  ({1000*t_total/n_water:.2f} ms/pixel, '\n      f'{1000*t_total/n_tiles:.0f} ms/tile, {n_tiles} tiles)')

## Noise calibration

For OE, the ideal median `chi2` is ≈ 1.  If it deviates significantly, adjust
`NOISE` and re-run: `NOISE_cal = NOISE × sqrt(median_chi2)`.

In [ ]:
chi2 = results['chi2']
valid_chi2 = chi2[np.isfinite(chi2)]
med_chi2   = float(np.median(valid_chi2))
noise_cal  = NOISE * np.sqrt(med_chi2)

print(f'NOISE = {NOISE:.4f} sr⁻¹')
print(f'Median chi2 = {med_chi2:.3f}  →  calibrated NOISE = {noise_cal:.4f} sr⁻¹')
if abs(med_chi2 - 1.0) > 0.15:
    print(f'→ Update NOISE = {noise_cal:.4f} and re-run.')
else:
    print('✓ chi2 ≈ 1 — noise well-calibrated.')

## Retrieved parameter maps

In [ ]:
x_hat     = results['x_hat']         # (n_rows, n_cols, n_fit)
sigma     = results['sigma']          # (n_rows, n_cols, n_fit)
A_diag    = results['A_diag']         # (n_rows, n_cols, n_fit)
H_info    = results['H_info']         # (n_rows, n_cols)
chi2_sp   = results['chi2_spectral']  # (n_rows, n_cols)
n_steps   = results['n_steps']        # (n_rows, n_cols)
fit_names = results['fit_names']

_style = {
    'C_Y':   ('YlOrBr', 0, 3,  'C_Y CDOM [1/m]'),
    'C_ism': ('Greys',  0, 50, 'C_ism ISM [g/m³]'),
}

n_params = len(fit_names)
ncols = 5
nrows = int(np.ceil(n_params / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes_flat = axes.ravel() if nrows > 1 else list(axes)

for ax, name in zip(axes_flat, fit_names):
    idx = fit_names.index(name)
    if name in _style:
        cmap, vmin, vmax, title = _style[name]
    else:
        cmap, vmin, vmax, title = 'YlGn', 0, 50, f'{name} [µg/L]'
    im = ax.imshow(x_hat[..., idx], cmap=cmap, vmin=vmin, vmax=vmax, origin='upper')
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

for ax in axes_flat[n_params:]:
    ax.axis('off')

plt.suptitle('bi_jax OE per-pixel — retrieved parameters', fontsize=12)
plt.tight_layout()
plt.show()

## OE diagnostics — chi2, H_info, A_diag, n_steps

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

im0 = axes[0, 0].imshow(chi2, cmap='RdYlGn_r', origin='upper',
                         vmin=0, vmax=np.nanpercentile(valid_chi2, 98))
plt.colorbar(im0, ax=axes[0, 0], shrink=0.85)
axes[0, 0].set_title(f'chi2 (target ≈ 1, median={med_chi2:.2f})')
axes[0, 0].axis('off')

chi2_sp_v = chi2_sp[np.isfinite(chi2_sp)]
im1 = axes[0, 1].imshow(chi2_sp, cmap='RdYlGn_r', origin='upper',
                         vmin=0, vmax=np.nanpercentile(chi2_sp_v, 98))
plt.colorbar(im1, ax=axes[0, 1], shrink=0.85)
axes[0, 1].set_title(f'chi2_spectral (plain MSE, median={np.nanmedian(chi2_sp_v):.2e})')
axes[0, 1].axis('off')

H_v = H_info[np.isfinite(H_info)]
im2 = axes[0, 2].imshow(H_info, cmap='viridis', origin='upper',
                         vmin=0, vmax=np.nanpercentile(H_v, 98))
plt.colorbar(im2, ax=axes[0, 2], shrink=0.85, label='nats')
axes[0, 2].set_title(f'H_info (information content, median={np.nanmedian(H_v):.2f} nats)')
axes[0, 2].axis('off')

im3 = axes[1, 0].imshow(A_diag.mean(axis=-1), cmap='viridis', origin='upper', vmin=0, vmax=1)
plt.colorbar(im3, ax=axes[1, 0], shrink=0.85)
axes[1, 0].set_title('mean A_diag (DFS/n_params)')
axes[1, 0].axis('off')

ns_v = n_steps[n_steps >= 0]
im4 = axes[1, 1].imshow(n_steps, cmap='plasma', origin='upper', vmin=0, vmax=MAX_STEPS)
plt.colorbar(im4, ax=axes[1, 1], shrink=0.85)
sat_pct = 100 * (ns_v == MAX_STEPS).mean()
axes[1, 1].set_title(f'n_steps (median={int(np.median(ns_v))}, '
                     f'saturated={sat_pct:.0f}%)')
axes[1, 1].axis('off')

axes[1, 2].hist(ns_v, bins=range(0, MAX_STEPS + 2), density=True,
                color='steelblue', alpha=0.8)
axes[1, 2].axvline(MAX_STEPS, color='r', ls='--', lw=1.5, label=f'cap={MAX_STEPS}')
axes[1, 2].set_xlabel('n_steps')
axes[1, 2].set_ylabel('Density')
axes[1, 2].set_title('n_steps distribution')
axes[1, 2].legend()

plt.suptitle('OE diagnostics', fontsize=12)
plt.tight_layout()
plt.show()

## Posterior uncertainty maps

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes_flat = axes.ravel() if nrows > 1 else list(axes)

for ax, name in zip(axes_flat, fit_names):
    idx   = fit_names.index(name)
    s     = sigma[..., idx]
    s_v   = s[np.isfinite(s)]
    im    = ax.imshow(s, cmap='Purples', origin='upper',
                      vmin=0, vmax=np.nanpercentile(s_v, 98))
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(f'σ({name})', fontsize=10)
    ax.axis('off')

for ax in axes_flat[n_params:]:
    ax.axis('off')

plt.suptitle('Posterior uncertainty σ per parameter', fontsize=12)
plt.tight_layout()
plt.show()

## Spot-check — observed vs fitted spectra

Sample 6 pixels: worst, best, and 4 percentile-spaced by `chi2_spectral`.

In [ ]:
y_hat  = results['y_hat']   # (n_rows, n_cols, n_obs)

chi2_flat = chi2_sp.ravel()
valid_px  = np.where(np.isfinite(chi2_flat))[0]
ranked    = valid_px[np.argsort(chi2_flat[valid_px])]

# best, 20th, 40th, 60th, 80th percentile, worst
picks = [ranked[int(len(ranked) * q)] for q in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]]
labels_picks = ['best', 'p20', 'p40', 'p60', 'p80', 'worst']

Rrs_flat  = Rrs_arr.reshape(-1, n_obs)
y_hat_flat = y_hat.reshape(-1, n_obs)

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=False)
for ax, px_idx, label in zip(axes.ravel(), picks, labels_picks):
    obs   = Rrs_flat[px_idx]
    fitted = y_hat_flat[px_idx]
    chi2_px = float(chi2_flat[px_idx])

    ax.plot(wavelengths, obs,    'k-',  lw=2, label='observed')
    ax.plot(wavelengths, fitted, 'r--', lw=2, label='fitted')
    ax.fill_between(wavelengths, obs, fitted, alpha=0.15, color='r')
    ax.set_title(f'{label} — chi2_sp={chi2_px:.2e}')
    ax.set_xlabel('λ [nm]')
    ax.set_ylabel('Rrs [sr⁻¹]')
    if label == 'best':
        ax.legend(fontsize=8)

plt.suptitle('Observed vs fitted spectra', fontsize=12)
plt.tight_layout()
plt.show()

## Save to S3 store (optional)

In [ ]:
# coords_2d = {'y': rrs.y, 'x': rrs.x}
# ds = dask_engine.to_dataset(
#     results, wavelengths=wavelengths,
#     coords={'y': rrs.y.values, 'x': rrs.x.values},
# ).rio.write_crs(scene_crs)
# store.write_data(ds, f'{OUTPUT_PREFIX}{scene_id}-wq_params_oe_dask_bi.zarr', replace=True)
# print('Saved.')